# Round 3: the pre-registered decisive test (Colab T4)

The plan, hypotheses and analysis are fixed in `docs/PREREGISTRATION.md`. **Commit and push it, the `configs/round3_*.yaml` files, `scripts/analyze_round3.py` and this notebook to a public repository (or upload them to OSF) before starting the first run.** The timestamp proves the plan came before the results.

| Part | What it answers | Runs | Time |
|---|---|---|---|
| A. Primary | Does M2-CL at the authors' strength (alpha 0.07) beat M2 at the paper's lr (1e-3)? | 8 M2 + 12 M2-CL | about 10 h |
| B. Secondary | Do M2 and M2-CL beat ERM at lr 1e-3? | 12 ERM | about 4-5 h |
| C. Replication | Does the answer hold on a second dataset (VLCS)? | 36 | about 15 h |
| D. Robustness | Seeds 3-4 for Part A | 16 | about 8 h |

**Platforms (pre-registration, Amendments 1-2):** A, B and D run on Colab; C runs only on the institute GPU machine (`scripts/run_part_c.py`). Never run the Part C cells here.

Run the parts in order: **A is the one that decides the paper**, so finish it before starting the others. Each part saves to its own folder in `My Drive/m2cl_v2/` (`lr/`, `round3/`, `round3_vlcs/`), and every run resumes from its last saved epoch.

**Before you start**, upload to the top level of My Drive:
1. The new `M2CL_code.zip` from the repo's `colab_upload/` folder (built 7 Nov 2026; it adds VLCS support). The setup cell stops if an older one is uploaded.
2. `PACS.zip`
3. Optional, so the results cell can run the full analysis: `lr.zip` (round 2) and `m2_v2.zip` (rounds 1-2).

Then choose Runtime > Change runtime type > **T4 GPU** and run the cells in order. Never change a setting in the training cells: the pre-registration fixes them.

**If Colab stops or this account runs out:** download the `m2cl_v2` folder from this account's Drive, upload the zip to the next account's My Drive together with the files above (don't rename it), and run all the cells again there. Unfinished runs continue where they stopped.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

DRIVE    = '/content/drive/MyDrive'
CODE_ZIP = f'{DRIVE}/M2CL_code.zip'
PACS_ZIP = f'{DRIVE}/PACS.zip'
OUT      = f'{DRIVE}/m2cl_v2'

import os, shutil, zipfile
import torch
assert torch.cuda.is_available(), 'No GPU: choose Runtime > Change runtime type > T4 GPU, then run this cell again.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
for path in (CODE_ZIP, PACS_ZIP):
    assert os.path.exists(path), f'Not found: {path}. Upload it to the top level of My Drive.'

shutil.rmtree('/content/repo', ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/repo')
os.chdir('/content/repo')
assert 'detect_domains' in open('src/m2cl/splits.py', encoding='utf-8').read() and os.path.exists('configs/round3_m2cl.yaml'), \
    'This M2CL_code.zip is older than round 3. Upload the new one from the colab_upload folder and run this cell again.'
!pip -q install pyyaml tabulate gdown

if not os.path.exists('/content/PACS_raw'):
    print('Extracting PACS...')
    zipfile.ZipFile(PACS_ZIP).extractall('/content/PACS_raw')

# Use the real images: the folder of four domains whose files are not dct_* (docs/AUDIT.md, A0)
def is_rgb(root):
    folder = os.path.join(root, 'art_painting', 'dog')
    return os.path.isdir(folder) and not any(f.startswith('dct_') for f in os.listdir(folder))
matches = [r for r, d, _ in os.walk('/content/PACS_raw')
           if {'art_painting', 'cartoon', 'photo', 'sketch'} <= set(d) and is_rgb(r)]
assert matches, 'No RGB copy of PACS found in PACS.zip.'
DATA = matches[0]
n_images = sum(len(f) for _, _, f in os.walk(DATA))
print('PACS root:', DATA, '| images:', n_images)
assert n_images == 9991, f'Expected 9,991 images, found {n_images}.'
os.makedirs(OUT, exist_ok=True)

free_gb = shutil.disk_usage(DRIVE).free / 1e9
print(f'Free Drive space: {free_gb:.1f} GB')
if free_gb < 5:
    print('WARNING: less than 5 GB free. Empty the Drive trash before training.')

In [ ]:
# 2. Only after switching accounts: copy round-3 progress out of an uploaded results zip
# (any name starting with m2cl_v2, e.g. m2cl_v2_round3_colab.zip, at the top of My Drive or one folder down).
# Nothing to do on the first account. Files already here are kept, so rerunning is safe.
import glob

drive.mount('/content/drive', force_remount=True)   # so a zip uploaded after cell 1 ran is seen

PARTS = ('lr', 'round3', 'round3_vlcs')
found = sorted(set(glob.glob(f'{DRIVE}/m2cl_v2*.zip') + glob.glob(f'{DRIVE}/*/m2cl_v2*.zip')),
               key=os.path.getmtime, reverse=True)
if not found:
    others = sorted(os.path.relpath(p, DRIVE) for p in glob.glob(f'{DRIVE}/*.zip') + glob.glob(f'{DRIVE}/*/*.zip'))
    print('No zip named m2cl_v2*.zip found in My Drive (top level or one folder down).')
    print('Zips that are there:', others or 'none')
    print('If the upload is still running, wait until Drive shows it complete, then run this cell again.')
    print("A file in 'Shared with me' is not in My Drive: download it and upload it to this account.")

copied = 0
for path in found:
    size_gb = os.path.getsize(path) / 1e9
    try:
        z = zipfile.ZipFile(path)
    except zipfile.BadZipFile:
        print(f'{os.path.relpath(path, DRIVE)} ({size_gb:.2f} GB): not a complete zip yet. '
              'Wait for the upload to finish, then run this cell again.')
        continue
    new = kept = 0
    with z:
        for info in z.infolist():
            parts = info.filename.replace('\\', '/').split('/')
            if info.filename.endswith('/') or parts[-1].endswith('.tmp'):
                continue
            starts = [i for i, p in enumerate(parts[:-1]) if p in PARTS]
            if not starts:
                continue
            target = os.path.join(OUT, *parts[starts[0]:])
            if os.path.exists(target):
                kept += 1
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target + '.tmp', 'wb') as dst:
                shutil.copyfileobj(src, dst)
            os.replace(target + '.tmp', target)
            new += 1
    copied += new
    print(f'{os.path.relpath(path, DRIVE)} ({size_gb:.2f} GB): copied {new} files, {kept} already here.')
print(f'Read {len(found)} zip(s); copied {copied} files. Now run the status cell.')

In [ ]:
# 3. Where each run stands. Run this any time.
import json
import pandas as pd

PACS_T = ('art_painting', 'cartoon', 'photo', 'sketch')
VLCS_T = ('Caltech101', 'LabelMe', 'SUN09', 'VOC2007')
PLAN = ([('A. M2 lr 1e-3', f'{t} seed {s}', f'{OUT}/lr/lr_0.001/m2_6tap_{t}_seed{s}') for t in PACS_T for s in (1, 2)]
        + [('A. M2-CL a0.07', f'{t} seed {s}', f'{OUT}/round3/m2cl/m2cl_6tap_{t}_seed{s}') for t in PACS_T for s in (0, 1, 2)]
        + [('B. ERM lr 1e-3', f'{t} seed {s}', f'{OUT}/round3/erm/erm_{t}_seed{s}') for t in PACS_T for s in (0, 1, 2)]
        + [(f'C. VLCS {m}', f'{t} seed {s}', f'{OUT}/round3_vlcs/{m}/{p}{t}_seed{s}')
           for m, p in (('m2', 'm2_6tap_'), ('m2cl', 'm2cl_6tap_'), ('erm', 'erm_')) for t in VLCS_T for s in (0, 1, 2)]
        + [('D. M2 seeds 3-4', f'{t} seed {s}', f'{OUT}/lr/lr_0.001/m2_6tap_{t}_seed{s}') for t in PACS_T for s in (3, 4)]
        + [('D. M2-CL seeds 3-4', f'{t} seed {s}', f'{OUT}/round3/m2cl/m2cl_6tap_{t}_seed{s}') for t in PACS_T for s in (3, 4)])

def run_status(run_dir):
    result = os.path.join(run_dir, 'result.json')
    if os.path.exists(result) and json.load(open(result)).get('status') == 'complete':
        return 'finished'
    for name in ('latest.pt', 'latest_prev.pt'):
        try:
            return f"continues at epoch {torch.load(os.path.join(run_dir, name), map_location='cpu', weights_only=False)['epoch'] + 1}"
        except Exception:
            pass
    return 'not started'

status = pd.DataFrame([{'part': p, 'run': r, 'status': run_status(d)} for p, r, d in PLAN])
summary = status.groupby('part', sort=False).status.apply(lambda s: f"{(s == 'finished').sum()} of {len(s)} finished")
display(summary.to_frame())

## A. Primary, pre-registered: M2 vs M2-CL at lr 1e-3, alpha 0.07, seeds 0-2
M2 seed 0 at lr 1e-3 already exists from round 2, so this trains M2 seeds 1-2 and M2-CL seeds 0-2. About 10 hours; it can span several sessions.

In [ ]:
!python scripts/train_lodo.py --config configs/round3_m2.yaml --seeds 1 2 --data_root "$DATA" --out_dir "$OUT/lr/lr_0.001"
!python scripts/train_lodo.py --config configs/round3_m2cl.yaml --data_root "$DATA" --out_dir "$OUT/round3/m2cl"

## B. Secondary: ERM at lr 1e-3, seeds 0-2

In [ ]:
!python scripts/train_lodo.py --config configs/round3_erm.yaml --data_root "$DATA" --out_dir "$OUT/round3/erm"

## C. Replication on VLCS
The next cell gets VLCS from the first source that works:
1. a `VLCS.tar.gz` at the top of My Drive: either uploaded, or added as a shortcut to DomainBed's file (open https://drive.google.com/file/d/1skwblH1_okBwxWxmRsp9_qi15hyPpxg8 while signed in to this account, then use Organize > Add shortcut > My Drive, and name the shortcut `VLCS.tar.gz`);
2. DomainBed's Google Drive link. It fails when Google's daily download limit for that file is reached ("Cannot retrieve the public link of the file");
3. a Hugging Face mirror of DomainBed's VLCS (`PTeterwak/DomainBed_OOP`, `all/VLCS.tar.gz`, 3.8 GB, only to this Colab's disk, not to your Drive).

Whatever the source, the cell checks that every domain has exactly DomainBed's image count (10,729 in total) and stops if not, so a different version of VLCS can't slip in.

In [ ]:
# VLCS: a copy in My Drive, then DomainBed's link, then the Hugging Face mirror.
import glob, tarfile
EXPECTED = {'Caltech101': 1415, 'LabelMe': 2656, 'SUN09': 3282, 'VOC2007': 3376}   # DomainBed: 10,729 images
IMG_EXTS = ('.jpg', '.jpeg', '.png')                                                 # as in src/m2cl/splits.py
DOMAINBED_ID = '1skwblH1_okBwxWxmRsp9_qi15hyPpxg8'
MIRROR = 'https://huggingface.co/datasets/PTeterwak/DomainBed_OOP/resolve/main/all/VLCS.tar.gz'

def vlcs_root():
    roots = [r for r, d, _ in os.walk('/content/VLCS_raw') if set(VLCS_T) <= set(d)]
    return roots[0] if roots else None

def image_counts(root):
    return {t: sum(f.lower().endswith(IMG_EXTS) for _, _, fs in os.walk(os.path.join(root, t)) for f in fs)
            for t in VLCS_T}

def extract_and_check(tar_path):
    if not os.path.exists(tar_path) or os.path.getsize(tar_path) < 100e6:
        print(f'  {tar_path}: missing or incomplete')
        return False
    print(f'  extracting {tar_path} ({os.path.getsize(tar_path) / 1e9:.1f} GB)...')
    shutil.rmtree('/content/VLCS_raw', ignore_errors=True)
    try:
        with tarfile.open(tar_path, 'r:*') as t:
            t.extractall('/content/VLCS_raw')
    except (tarfile.TarError, EOFError, OSError) as e:
        print(f'  could not extract it: {e}')
        return False
    root = vlcs_root()
    counts = image_counts(root) if root else None
    if counts != EXPECTED:
        print(f'  wrong contents: {counts}, expected {EXPECTED}')
        return False
    return True

ready = vlcs_root() is not None and image_counts(vlcs_root()) == EXPECTED
if not ready:
    print('1. Looking for VLCS in My Drive...')
    for path in sorted(glob.glob(f'{DRIVE}/VLCS*.tar.gz') + glob.glob(f'{DRIVE}/VLCS*.tar')):
        ready = extract_and_check(path)
        if ready:
            break
if not ready:
    print("2. Trying DomainBed's Google Drive link...")
    get_ipython().system('pip -q install -U gdown')
    if os.path.exists('/content/VLCS.tar.gz'):
        os.remove('/content/VLCS.tar.gz')
    get_ipython().system(f'gdown {DOMAINBED_ID} -O /content/VLCS.tar.gz')
    ready = extract_and_check('/content/VLCS.tar.gz')
if not ready:
    print('3. Downloading the Hugging Face mirror (3.8 GB)...')
    if os.path.exists('/content/VLCS_mirror.tar.gz'):
        os.remove('/content/VLCS_mirror.tar.gz')
    get_ipython().system(f'wget -q --show-progress -O /content/VLCS_mirror.tar.gz {MIRROR}')
    ready = extract_and_check('/content/VLCS_mirror.tar.gz')
assert ready, ('No source gave the expected VLCS. Add the shortcut described above (or upload VLCS.tar.gz) '
               'to the top of My Drive and run this cell again.')
for leftover in ('/content/VLCS.tar.gz', '/content/VLCS_mirror.tar.gz'):
    if os.path.exists(leftover):
        os.remove(leftover)   # free the Colab disk; the extracted copy stays in /content/VLCS_raw
VLCS = vlcs_root()
print('VLCS root:', VLCS, '| images per domain:', image_counts(VLCS), '| total:', sum(image_counts(VLCS).values()))

In [ ]:
VT = 'Caltech101 LabelMe SUN09 VOC2007'
!python scripts/train_lodo.py --config configs/round3_m2.yaml --targets $VT --data_root "$VLCS" --out_dir "$OUT/round3_vlcs/m2"
!python scripts/train_lodo.py --config configs/round3_m2cl.yaml --targets $VT --data_root "$VLCS" --out_dir "$OUT/round3_vlcs/m2cl"
!python scripts/train_lodo.py --config configs/round3_erm.yaml --targets $VT --data_root "$VLCS" --out_dir "$OUT/round3_vlcs/erm"

## D. Robustness: Part A with seeds 3-4
Reported separately; never pooled into the primary test.

In [ ]:
!python scripts/train_lodo.py --config configs/round3_m2.yaml --seeds 3 4 --data_root "$DATA" --out_dir "$OUT/lr/lr_0.001"
!python scripts/train_lodo.py --config configs/round3_m2cl.yaml --seeds 3 4 --data_root "$DATA" --out_dir "$OUT/round3/m2cl"

In [ ]:
# 4. Pre-registered analysis of everything finished so far
SOURCES = ' '.join(f'"{p}"' for p in [OUT] + [f'{DRIVE}/{z}' for z in ('lr.zip', 'm2_v2.zip') if os.path.exists(f'{DRIVE}/{z}')])
!python scripts/analyze_round3.py $SOURCES --json "$OUT/round3_analysis.json"

## When you're done
Download `My Drive/m2cl_v2/lr`, `round3`, `round3_vlcs` (or the whole `m2cl_v2` folder) and `round3_analysis.json` to the laptop, and tell Claude which runs printed a `[resume]` line. Claude then runs `scripts/analyze_round3.py` with `--grad` for the manipulation check and writes the results into the paper.